# RelBench rel-avito Dataset: FK Links, Cardinality Stats & Parent-Child Feature Matrices

This notebook demonstrates the **RelBench rel-avito dataset** (NeurIPS 2024 benchmark) extraction pipeline.

**What it does:**
- Loads pre-computed aligned parent-child feature pairs extracted from the Avito ad platform dataset (8 tables, ~20.7M rows)
- Explores 11 foreign-key (FK) links with cardinality statistics (mean/median/max/p95/std)
- Each example is one `(parent_features, child_features)` pair joined on a foreign key
- Analyzes feature distributions and computes R² between parent and child feature vectors
- Visualizes cardinality statistics and feature correlations across FK links

**Source:** [RelBench](https://relbench.stanford.edu/) (Stanford SNAP) — Avito online advertisement platform dataset.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'tabulate==0.9.0')

In [ ]:
import json
import math
import os
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Load Data

Load the pre-computed mini dataset from GitHub (with local fallback).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/dataset_iter4_relbench_rel_av/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Dataset: {data['metadata']['dataset_name']}")
print(f"Source: {data['metadata']['source']}")
print(f"Domain: {data['metadata']['domain']}")
print(f"Tables: {len(data['metadata']['tables'])}")
print(f"FK links: {len(data['metadata']['fk_links'])}")
print(f"Tasks: {len(data['metadata']['tasks'])}")
print(f"Examples: {len(data['datasets'][0]['examples'])}")

## Configuration

Tunable parameters for the analysis. These mirror the original script's configuration.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
# Original script values (for reference):
# MAX_SAMPLE_ROWS = 500_000
# MAX_ALIGNED_PAIRS = 5_000
# MAX_FEATURE_COLS = 15

# Demo parameters
MAX_DISPLAY_EXAMPLES = 5       # examples to show in detail per FK link
MAX_DISPLAY_LINKS = 11         # FK links to analyze (all available)
MAX_FEATURE_COLS = 15          # cap features per side (matches original)

## Helper Functions

These mirror the original `data.py` utility functions for processing features.

In [ ]:
def safe_float(val) -> float:
    """Convert value to a JSON-safe float."""
    if val is None or (isinstance(val, float) and (math.isnan(val) or math.isinf(val))):
        return 0.0
    return float(val)


def round_list(vals: list, decimals: int = 4) -> list:
    """Round floats in a list to reduce JSON size."""
    return [round(v, decimals) if isinstance(v, float) else v for v in vals]

## Explore Table Metadata

The rel-avito dataset consists of 8 tables from the Avito ad platform. Let's inspect the table structure.

In [ ]:
tables = data["metadata"]["tables"]
table_rows = []
for tname, tmeta in tables.items():
    fk_count = len(tmeta.get("fkey_col_to_pkey_table", {}))
    table_rows.append({
        "Table": tname,
        "Rows": f"{tmeta['num_rows']:,}",
        "Cols": tmeta["num_cols"],
        "PKey": tmeta["pkey_col"] or "—",
        "Time Col": tmeta.get("time_col") or "—",
        "FK Count": fk_count,
    })

df_tables = pd.DataFrame(table_rows)
print("=== Avito Dataset Tables ===")
print(df_tables.to_string(index=False))
print(f"\nTotal rows across all tables: {sum(t['num_rows'] for t in tables.values()):,}")

## FK Link Cardinality Statistics

Each FK link connects a parent table to a child table via a foreign key. Cardinality describes how many child rows each parent key maps to.

In [ ]:
fk_links = data["metadata"]["fk_links"]
link_rows = []
for link_id, info in list(fk_links.items())[:MAX_DISPLAY_LINKS]:
    link_rows.append({
        "FK Link": info["fk_link"],
        "FK Column": info["fk_column"],
        "Pairs": info["num_aligned_pairs"],
        "Card Mean": round(info["cardinality_mean"], 1),
        "Card Median": round(info["cardinality_median"], 1),
        "Card Max": info["cardinality_max"],
        "Card P95": round(info["cardinality_p95"], 1),
        "Card Std": round(info["cardinality_std"], 1),
        "Parent Feats": len(info["parent_feature_cols"]),
        "Child Feats": len(info["child_feature_cols"]),
    })

df_links = pd.DataFrame(link_rows)
print("=== FK Link Cardinality Statistics ===")
print(df_links.to_string(index=False))
print(f"\nTotal FK links: {len(fk_links)}")

## Benchmark Tasks

The rel-avito dataset includes 6 benchmark tasks: regression, binary classification, and link prediction.

In [ ]:
tasks = data["metadata"]["tasks"]
task_rows = []
for t in tasks:
    task_rows.append({
        "Task": t["task_name"],
        "Type": t.get("task_type", "—"),
        "Entity Table": t.get("entity_table", "—"),
        "Target Col": t.get("target_col", "—"),
        "Metrics": ", ".join(t.get("metrics", [])),
    })

df_tasks = pd.DataFrame(task_rows)
print("=== Benchmark Tasks ===")
print(df_tasks.to_string(index=False))

## Analyze Parent-Child Feature Pairs

Each example contains aligned `(parent_features, child_features)` vectors joined on a foreign key. Let's parse them and compute basic statistics per FK link.

In [ ]:
examples = data["datasets"][0]["examples"]

# Group examples by FK link
by_link = defaultdict(list)
for ex in examples:
    by_link[ex["metadata_link_id"]].append(ex)

print(f"Total examples: {len(examples)}")
print(f"FK links represented: {len(by_link)}")
print()

# Parse feature vectors and compute per-link statistics
link_stats = {}
for link_id, exs in list(by_link.items())[:MAX_DISPLAY_LINKS]:
    parent_vecs = [json.loads(e["input"]) for e in exs]
    child_vecs = [json.loads(e["output"]) for e in exs]

    parent_arr = np.array(parent_vecs, dtype=np.float64)
    child_arr = np.array(child_vecs, dtype=np.float64)

    # Compute per-feature mean and std for parent and child
    p_mean = np.nanmean(parent_arr, axis=0)
    p_std = np.nanstd(parent_arr, axis=0)
    c_mean = np.nanmean(child_arr, axis=0)
    c_std = np.nanstd(child_arr, axis=0)

    # Compute R² between flattened parent and child feature vectors
    # (measures how well parent features predict child features overall)
    p_flat = parent_arr.flatten()
    c_flat = child_arr.flatten()
    # Pad shorter to match lengths if needed
    min_len = min(len(p_flat), len(c_flat))
    p_flat, c_flat = p_flat[:min_len], c_flat[:min_len]

    ss_res = np.sum((c_flat - p_flat) ** 2)
    ss_tot = np.sum((c_flat - np.mean(c_flat)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    r2 = safe_float(r2)

    link_stats[link_id] = {
        "n_examples": len(exs),
        "parent_dims": parent_arr.shape[1],
        "child_dims": child_arr.shape[1],
        "parent_mean": round_list(p_mean.tolist()),
        "parent_std": round_list(p_std.tolist()),
        "child_mean": round_list(c_mean.tolist()),
        "child_std": round_list(c_std.tolist()),
        "r2_flat": round(r2, 4),
    }

    fk_label = exs[0]["metadata_fk_link"]
    print(f"  {fk_label} ({link_id}):")
    print(f"    Examples: {len(exs)}, Parent dims: {parent_arr.shape[1]}, Child dims: {child_arr.shape[1]}")
    print(f"    Parent mean: {round_list(p_mean.tolist())}")
    print(f"    Child mean:  {round_list(c_mean.tolist())}")
    print(f"    R² (flat):   {r2:.4f}")
    print()

## Sample Examples

Display a few raw examples to see the parent-child feature pair structure.

In [ ]:
# Show a few examples from each FK link
for link_id, exs in list(by_link.items())[:3]:
    fk_label = exs[0]["metadata_fk_link"]
    info = fk_links.get(link_id, {})
    parent_cols = info.get("parent_feature_cols", [])
    child_cols = info.get("child_feature_cols", [])

    print(f"--- {fk_label} ---")
    print(f"    Parent features: {parent_cols}")
    print(f"    Child features:  {child_cols}")
    for ex in exs[:MAX_DISPLAY_EXAMPLES]:
        parent_feats = json.loads(ex["input"])
        child_feats = json.loads(ex["output"])
        print(f"    Row {ex['metadata_row_index']}: parent={parent_feats}  child={child_feats}")
    print()

## Visualization

Plot cardinality statistics across FK links, R² scores, and feature distributions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── Plot 1: Cardinality mean vs max (log scale) per FK link ──
ax = axes[0, 0]
link_labels = []
card_means = []
card_maxes = []
card_medians = []
for link_id, info in fk_links.items():
    link_labels.append(info["fk_link"].replace("->", "\n→"))
    card_means.append(info["cardinality_mean"])
    card_maxes.append(info["cardinality_max"])
    card_medians.append(info["cardinality_median"])

x = np.arange(len(link_labels))
width = 0.35
ax.bar(x - width/2, card_means, width, label="Mean", color="steelblue")
ax.bar(x + width/2, card_medians, width, label="Median", color="coral")
ax.set_yscale("log")
ax.set_ylabel("Cardinality (log scale)")
ax.set_title("FK Link Cardinality: Mean vs Median")
ax.set_xticks(x)
ax.set_xticklabels(link_labels, rotation=45, ha="right", fontsize=7)
ax.legend()

# ── Plot 2: R² scores per FK link ──
ax = axes[0, 1]
r2_labels = []
r2_values = []
for link_id, stats in link_stats.items():
    fk_label = by_link[link_id][0]["metadata_fk_link"]
    r2_labels.append(fk_label.replace("->", "\n→"))
    r2_values.append(stats["r2_flat"])

colors = ["green" if v > 0 else "red" for v in r2_values]
ax.barh(range(len(r2_labels)), r2_values, color=colors, alpha=0.7)
ax.set_yticks(range(len(r2_labels)))
ax.set_yticklabels(r2_labels, fontsize=7)
ax.set_xlabel("R² (flat)")
ax.set_title("R² Between Parent & Child Features")
ax.axvline(x=0, color="black", linewidth=0.5)

# ── Plot 3: Table sizes ──
ax = axes[1, 0]
table_names = list(tables.keys())
table_sizes = [tables[t]["num_rows"] for t in table_names]
bars = ax.barh(table_names, table_sizes, color="teal", alpha=0.7)
ax.set_xscale("log")
ax.set_xlabel("Number of Rows (log scale)")
ax.set_title("Table Sizes")
for bar, size in zip(bars, table_sizes):
    ax.text(bar.get_width(), bar.get_y() + bar.get_height()/2,
            f" {size:,}", va="center", fontsize=8)

# ── Plot 4: Feature dimensionality per FK link ──
ax = axes[1, 1]
parent_dims = []
child_dims = []
dim_labels = []
for link_id, info in fk_links.items():
    dim_labels.append(info["fk_link"].replace("->", "\n→"))
    parent_dims.append(len(info["parent_feature_cols"]))
    child_dims.append(len(info["child_feature_cols"]))

x = np.arange(len(dim_labels))
ax.bar(x - width/2, parent_dims, width, label="Parent Feats", color="mediumpurple")
ax.bar(x + width/2, child_dims, width, label="Child Feats", color="goldenrod")
ax.set_ylabel("Number of Features")
ax.set_title("Feature Dimensionality per FK Link")
ax.set_xticks(x)
ax.set_xticklabels(dim_labels, rotation=45, ha="right", fontsize=7)
ax.legend()

plt.tight_layout()
plt.savefig("fk_analysis.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: fk_analysis.png")

## Summary Results

Final summary of the FK link analysis with all computed statistics.

In [ ]:
# ── Summary Table ──
summary_rows = []
for link_id, stats in link_stats.items():
    info = fk_links.get(link_id, {})
    summary_rows.append({
        "FK Link": info.get("fk_link", link_id),
        "Examples": stats["n_examples"],
        "Parent Dims": stats["parent_dims"],
        "Child Dims": stats["child_dims"],
        "Card Mean": round(info.get("cardinality_mean", 0), 1),
        "Card Max": info.get("cardinality_max", 0),
        "R² (flat)": stats["r2_flat"],
    })

df_summary = pd.DataFrame(summary_rows)
print("=" * 80)
print("SUMMARY: FK Link Analysis for rel-avito")
print("=" * 80)
print(df_summary.to_string(index=False))
print()
print(f"Dataset: {data['metadata']['dataset_name']}")
print(f"Total tables: {len(tables)}")
print(f"Total FK links analyzed: {len(link_stats)}")
print(f"Total examples: {len(examples)}")
print(f"Benchmark tasks: {len(tasks)}")
print(f"Total rows across all tables: {sum(t['num_rows'] for t in tables.values()):,}")